# Multi-asset replication: ETH/USDT and SOL/USDT

The two setups in `SETUPS` (pullback in an uptrend; volume-confirmed continuation) were designed and pre-registered on BTC/USDT, and evaluated and backtested on BTC in notebooks 06 and 07. ETH/USDT and SOL/USDT were not used to design, tune or evaluate them, so their histories are out-of-sample evidence.

This notebook applies the existing modules exactly as they are: the same rules, thresholds, holding periods (5 and 10 days) and costs (0.15% per side). Nothing is changed in response to the results. Each asset is evaluated over its full available history, using the methods of notebook 06 (setup evaluation) and notebook 07 (backtests and random-entry baselines).

No predictions were recorded before these results were computed.

## Step 0: Imports and settings

`HORIZONS`, `COST` and `N_RUNS` are the values used in notebooks 06 and 07. `SETUPS` is imported from `cryptosense.setups` and printed unchanged.

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd

from cryptosense.data import load_processed, symbol_to_slug
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes
from cryptosense.setups import SETUPS, setup_flag
from cryptosense.backtest import run_backtest, summarize_backtest

DATA_DIR = Path.cwd().parent / "data"
SYMBOLS = ["ETH/USDT", "SOL/USDT"]
HORIZONS = [5, 10]
COST = 0.0015
N_RUNS = 2000

SETUPS

{'pullback': [('dist_sma_100', '>', 0.0), ('dist_sma_20', '<', 0.0)],
 'continuation': [('mom_20', '>', 0.0),
  ('er_20', '>', 0.4),
  ('rel_volume_20', '>', 1.5)]}

## Step 1: Load data, add features and regimes

Each asset is loaded with `load_processed`, then passed through `add_features` and `add_regimes`. If a processed file is missing, the notebook stops: click **Refresh data from Binance** in the dashboard, then run this notebook again.

In [3]:
missing = [
    symbol for symbol in SYMBOLS
    if not (DATA_DIR / "processed" / f"{symbol_to_slug(symbol)}_1d.csv").exists()
]
if missing:
    raise FileNotFoundError(
        f"Processed data missing for {', '.join(missing)}. "
        'Click "Refresh data from Binance" in the dashboard, then run this notebook again.'
    )

frames = {
    symbol: add_regimes(add_features(load_processed(symbol, DATA_DIR)))
    for symbol in SYMBOLS
}
for symbol, df in frames.items():
    print(f"{symbol}: {df.index[0].date()} -> {df.index[-1].date()} | {len(df)} days")

ETH/USDT: 2017-08-17 -> 2026-10-20 | 3352 days
SOL/USDT: 2020-08-11 -> 2026-10-20 | 2262 days


## Step 2: Setup evaluation over the full history (method of notebook 06)

Following notebook 06:
- Flags come from `setup_flag` and episode starts (first day of each run of flagged days) are computed on the full history.
- Outcomes are 5- and 10-day forward log returns of the close.
- The evaluation keeps only days on which every setup feature is defined and both outcomes are known.
- Standard error = standard deviation of the episode outcomes / √(number of episodes), as in notebook 06.

Two comparisons are made for each setup and horizon:
- **All days:** the mean over every evaluated day.
- **Same trend regime:** each episode is compared with the mean outcome of all evaluated days in the trend regime ("up", "down" or "range") of its start day, the base rate used by `explain_setup`; the table reports the average of these per-episode base rates.

In [4]:
FEATURES = sorted({feature for rules in SETUPS.values() for feature, _, _ in rules})
OUTCOMES = [f"fwd_ret_{h}" for h in HORIZONS]

def evaluation_frame(df):
    out = df.copy()
    for h in HORIZONS:
        out[f"fwd_ret_{h}"] = np.log(out["close"].shift(-h) / out["close"])
    for name in SETUPS:
        flag = setup_flag(out, name)
        out[f"{name}_start"] = flag & ~flag.shift(fill_value=False)
    return out.dropna(subset=FEATURES + ["trend"] + OUTCOMES)

def summarise(frame, label):
    row = {"group": label, "n": len(frame)}
    for h in HORIZONS:
        r = frame[f"fwd_ret_{h}"]
        row[f"mean_{h}d"] = r.mean()
        row[f"median_{h}d"] = r.median()
        row[f"positive_{h}d"] = (r > 0).mean()
    return row

evals = {symbol: evaluation_frame(df) for symbol, df in frames.items()}

summary_rows, comparison_rows = [], []
for symbol, ev in evals.items():
    print(f"{symbol}: evaluated {ev.index[0].date()} -> {ev.index[-1].date()} | {len(ev)} days")
    summary_rows.append({"asset": symbol, **summarise(ev, "all days")})
    for name in SETUPS:
        entries = ev[ev[f"{name}_start"]]
        summary_rows.append({"asset": symbol, **summarise(entries, f"{name} entries")})
        for h in HORIZONS:
            col = f"fwd_ret_{h}"
            r = entries[col]
            se = r.std() / np.sqrt(len(r))
            base_all = ev[col].mean()
            base_same = entries["trend"].map(ev.groupby("trend")[col].mean()).mean()
            comparison_rows.append({
                "asset": symbol,
                "setup": name,
                "horizon": h,
                "episodes": len(r),
                "mean_entries": r.mean(),
                "se": se,
                "mean_all_days": base_all,
                "diff_all": r.mean() - base_all,
                "diff_all / se": (r.mean() - base_all) / se,
                "mean_same_trend": base_same,
                "diff_same_trend": r.mean() - base_same,
                "diff_same_trend / se": (r.mean() - base_same) / se,
            })

setup_summary = pd.DataFrame(summary_rows).set_index(["asset", "group"])
setup_comparison = pd.DataFrame(comparison_rows).set_index(["asset", "setup", "horizon"])
setup_summary.round(4)

ETH/USDT: evaluated 2017-11-24 -> 2026-10-10 | 3243 days
SOL/USDT: evaluated 2020-11-18 -> 2026-10-10 | 2153 days


n  mean_5d  median_5d  positive_5d  \
asset    group                                                         
ETH/USDT all days              3243   0.0026     0.0027       0.5183   
         pullback entries       130   0.0104     0.0063       0.5615   
         continuation entries    60   0.0146    -0.0006       0.5000   
SOL/USDT all days              2153   0.0091     0.0024       0.5072   
         pullback entries        69  -0.0003    -0.0036       0.4783   
         continuation entries    35   0.0856     0.0742       0.7143   

                               mean_10d  median_10d  positive_10d  
asset    group                                                     
ETH/USDT all days                0.0052      0.0056        0.5211  
         pullback entries        0.0271     -0.0023        0.4923  
         continuation entries    0.0146      0.0070        0.5500  
SOL/USDT all days                0.0181      0.0042        0.5095  
         pullback entries       -0.0118     -0.0313        0.4348  
         continuation entries    0.1319      0.1335        0.6857

In [5]:
setup_comparison.round(4)

episodes  mean_entries      se  mean_all_days  \
asset    setup        horizon                                                  
ETH/USDT pullback     5             130        0.0104  0.0092         0.0026   
                      10            130        0.0271  0.0133         0.0052   
         continuation 5              60        0.0146  0.0121         0.0026   
                      10             60        0.0146  0.0175         0.0052   
SOL/USDT pullback     5              69       -0.0003  0.0123         0.0091   
                      10             69       -0.0118  0.0179         0.0181   
         continuation 5              35        0.0856  0.0202         0.0091   
                      10             35        0.1319  0.0331         0.0181   

                               diff_all  diff_all / se  mean_same_trend  \
asset    setup        horizon                                             
ETH/USDT pullback     5          0.0079         0.8494           0.0070   
                      10         0.0219         1.6465           0.0136   
         continuation 5          0.0121         0.9975           0.0117   
                      10         0.0094         0.5375           0.0235   
SOL/USDT pullback     5         -0.0093        -0.7565           0.0148   
                      10        -0.0299        -1.6721           0.0288   
         continuation 5          0.0766         3.7833           0.0246   
                      10         0.1137         3.4323           0.0525   

                               diff_same_trend  diff_same_trend / se  
asset    setup        horizon                                         
ETH/USDT pullback     5                 0.0035                0.3733  
                      10                0.0135                1.0149  
         continuation 5                 0.0029                0.2429  
                      10               -0.0089               -0.5083  
SOL/USDT pullback     5                -0.0150               -1.2204  
                      10               -0.0407               -2.2713  
         continuation 5                 0.0610                3.0159  
                      10                0.0794                2.3958

### Episodes by trend regime at the start day

For each setup, the number of episodes starting in each trend regime, their mean 5-day outcome, and the mean 5-day outcome of all evaluated days in that regime.

In [6]:
rows = []
for symbol, ev in evals.items():
    trend_base = ev.groupby("trend")["fwd_ret_5"].agg(days="size", base_5d="mean")
    for name in SETUPS:
        entries = ev[ev[f"{name}_start"]]
        table = entries.groupby("trend")["fwd_ret_5"].agg(episodes="size", mean_5d="mean")
        table = table.join(trend_base)
        table["excess_5d"] = table["mean_5d"] - table["base_5d"]
        table.index = pd.MultiIndex.from_product([[symbol], [name], table.index], names=["asset", "setup", "trend"])
        rows.append(table)

by_trend = pd.concat(rows)
by_trend.round(4)

episodes  mean_5d  days  base_5d  excess_5d
asset    setup        trend                                             
ETH/USDT pullback     range        64  -0.0051   907   0.0015    -0.0067
                      up           66   0.0255  1236   0.0122     0.0133
         continuation range         3   0.0064   907   0.0015     0.0048
                      up           57   0.0151  1236   0.0122     0.0028
SOL/USDT pullback     range        34  -0.0144   545   0.0033    -0.0178
                      up           35   0.0135   803   0.0259    -0.0124
         continuation range         2   0.0371   545   0.0033     0.0338
                      up           33   0.0886   803   0.0259     0.0627

## Step 3: Backtests over the full history (method of notebook 07)

Following notebook 07: the backtest frame keeps days where `dist_sma_100`, `rel_volume_20` and `er_20` are defined; signals come from `setup_flag`; `run_backtest` enters at the next day's open and exits at the open `hold` days later, one position at a time, with `cost_per_side=0.0015` (0.30% per round trip). Strategies: continuation (5- and 10-day holds), pullback as a control (5- and 10-day holds), and buy-and-hold (one trade from the first to the last open, same costs).

In [7]:
backtest_frames = {
    symbol: df.dropna(subset=["dist_sma_100", "rel_volume_20", "er_20"])
    for symbol, df in frames.items()
}

rows = []
for symbol, bt in backtest_frames.items():
    print(f"{symbol}: backtest {bt.index[0].date()} -> {bt.index[-1].date()} | {len(bt)} days")
    for setup in ["continuation", "pullback"]:
        signal = setup_flag(bt, setup)
        for hold in HORIZONS:
            trades, equity = run_backtest(bt, signal, hold, COST)
            rows.append({"asset": symbol, "strategy": f"{setup} {hold}d", **summarize_backtest(trades, equity)})

    bh_signal = pd.Series(False, index=bt.index)
    bh_signal.iloc[0] = True
    trades, equity = run_backtest(bt, bh_signal, len(bt) - 2, COST)
    rows.append({"asset": symbol, "strategy": "buy and hold", **summarize_backtest(trades, equity)})

backtest_results = pd.DataFrame(rows).set_index(["asset", "strategy"])
backtest_results.round(3)

ETH/USDT: backtest 2017-11-24 -> 2026-10-20 | 3253 days
SOL/USDT: backtest 2020-11-18 -> 2026-10-20 | 2163 days


total_return  annual_volatility  max_drawdown  \
asset    strategy                                                          
ETH/USDT continuation 5d          1.291              0.254        -0.282   
         continuation 10d         0.332              0.344        -0.550   
         pullback 5d              0.468              0.404        -0.661   
         pullback 10d             5.733              0.489        -0.676   
         buy and hold             4.135              0.853        -0.940   
SOL/USDT continuation 5d         12.898              0.359        -0.191   
         continuation 10d        20.619              0.442        -0.299   
         pullback 5d             -0.406              0.463        -0.813   
         pullback 10d            -0.395              0.533        -0.648   
         buy and hold            48.600              1.095        -0.963   

                           trades  time_in_market  win_rate  avg_trade  
asset    strategy                                                       
ETH/USDT continuation 5d       51           0.078     0.569      0.021  
         continuation 10d      39           0.120     0.513      0.020  
         pullback 5d          139           0.214     0.532      0.008  
         pullback 10d          94           0.289     0.489      0.035  
         buy and hold           1           1.000     1.000      4.135  
SOL/USDT continuation 5d       29           0.067     0.759      0.103  
         continuation 10d      23           0.106     0.696      0.166  
         pullback 5d           75           0.173     0.413     -0.001  
         pullback 10d          50           0.231     0.480     -0.002  
         buy and hold           1           1.000     1.000     48.600

### Random-entry baseline for the continuation strategy

As in notebook 07: for each holding period, the continuation strategy's average net trade is compared with 2,000 random sets of the same number of trades, drawn without replacement from trades signalled on any day, or only on up-trend days. Each random trade buys at the next open and sells `hold` days later with the same costs. `share_random_beating_actual` is the share of random draws whose average trade was at least as high as the strategy's.

In [8]:
def net_trade_returns(frame, hold, cost):
    """Net return of a trade signalled on each day: buy next open, sell `hold` days later."""
    entry = frame["open"].shift(-1)
    exit_ = frame["open"].shift(-(1 + hold))
    return ((exit_ * (1 - cost)) / (entry * (1 + cost)) - 1).dropna()

rng = np.random.default_rng(0)
rows = []
for symbol, bt in backtest_frames.items():
    for hold in HORIZONS:
        trades, _ = run_backtest(bt, setup_flag(bt, "continuation"), hold, COST)
        actual, k = trades["net_return"].mean(), len(trades)

        all_days = net_trade_returns(bt, hold, COST)
        up_days = all_days[bt.loc[all_days.index, "trend"] == "up"]

        for label, pool in [("any day", all_days), ("up-trend day", up_days)]:
            sims = np.array([
                rng.choice(pool.to_numpy(), size=k, replace=False).mean()
                for _ in range(N_RUNS)
            ])
            rows.append({
                "asset": symbol,
                "hold": hold,
                "random pool": label,
                "pool size": len(pool),
                "trades": k,
                "actual_avg_trade": actual,
                "random_median": np.median(sims),
                "share_random_beating_actual": (sims >= actual).mean(),
            })

random_baseline = pd.DataFrame(rows)
print(random_baseline.round(4).to_string())

      asset  hold   random pool  pool size  trades  actual_avg_trade  random_median  share_random_beating_actual
0  ETH/USDT     5       any day       3247      51            0.0211         0.0051                       0.1055
1  ETH/USDT     5  up-trend day       1236      51            0.0211         0.0141                       0.3075
2  ETH/USDT    10       any day       3242      39            0.0200         0.0124                       0.3820
3  ETH/USDT    10  up-trend day       1236      39            0.0200         0.0325                       0.6890
4  SOL/USDT     5       any day       2157      29            0.1028         0.0135                       0.0005
5  SOL/USDT     5  up-trend day        803      29            0.1028         0.0329                       0.0045
6  SOL/USDT    10       any day       2152      23            0.1661         0.0290                       0.0045
7  SOL/USDT    10  up-trend day        803      23            0.1661         0.0751             

## Results

Data: ETH/USDT 2017-08-17 to 2026-10-20 (3,352 days); SOL/USDT 2020-08-11 to 2026-10-20 (2,262 days). Setup evaluation covers days with all setup features and both outcomes known (ETH 2017-11-24 to 2026-10-10, 3,243 days; SOL 2020-11-18 to 2026-10-10, 2,153 days). Backtests cover ETH 2017-11-24 to 2026-10-20 (3,253 days) and SOL 2020-11-18 to 2026-10-20 (2,163 days). Setup returns are log returns of the close; backtest returns are simple returns after 0.15% costs per side.

### Setup evaluation: episode outcomes

| Asset | Group | n | Mean 5d | Median 5d | Positive 5d | Mean 10d | Median 10d | Positive 10d |
|---|---|---|---|---|---|---|---|---|
| ETH | all days | 3243 | +0.26% | +0.27% | 51.8% | +0.52% | +0.56% | 52.1% |
| ETH | pullback entries | 130 | +1.04% | +0.63% | 56.2% | +2.71% | −0.23% | 49.2% |
| ETH | continuation entries | 60 | +1.46% | −0.06% | 50.0% | +1.46% | +0.70% | 55.0% |
| SOL | all days | 2153 | +0.91% | +0.24% | 50.7% | +1.81% | +0.42% | 50.9% |
| SOL | pullback entries | 69 | −0.03% | −0.36% | 47.8% | −1.18% | −3.13% | 43.5% |
| SOL | continuation entries | 35 | +8.56% | +7.42% | 71.4% | +13.19% | +13.35% | 68.6% |

### Setup evaluation: comparison with base rates

| Asset | Setup | Horizon | Episodes | Mean entries | SE | All days | Diff | Diff / SE | Same trend | Diff | Diff / SE |
|---|---|---|---|---|---|---|---|---|---|---|---|
| ETH | pullback | 5d | 130 | +1.04% | 0.92% | +0.26% | +0.79% | +0.85 | +0.70% | +0.35% | +0.37 |
| ETH | pullback | 10d | 130 | +2.71% | 1.33% | +0.52% | +2.19% | +1.65 | +1.36% | +1.35% | +1.01 |
| ETH | continuation | 5d | 60 | +1.46% | 1.21% | +0.26% | +1.21% | +1.00 | +1.17% | +0.29% | +0.24 |
| ETH | continuation | 10d | 60 | +1.46% | 1.75% | +0.52% | +0.94% | +0.54 | +2.35% | −0.89% | −0.51 |
| SOL | pullback | 5d | 69 | −0.03% | 1.23% | +0.91% | −0.93% | −0.76 | +1.48% | −1.50% | −1.22 |
| SOL | pullback | 10d | 69 | −1.18% | 1.79% | +1.81% | −2.99% | −1.67 | +2.88% | −4.07% | −2.27 |
| SOL | continuation | 5d | 35 | +8.56% | 2.02% | +0.91% | +7.66% | +3.78 | +2.46% | +6.10% | +3.02 |
| SOL | continuation | 10d | 35 | +13.19% | 3.31% | +1.81% | +11.37% | +3.43 | +5.25% | +7.94% | +2.40 |

### Episodes by trend regime at the start day (5-day outcomes)

| Asset | Setup | Trend | Episodes | Mean 5d | Days in regime | Regime base 5d | Excess 5d |
|---|---|---|---|---|---|---|---|
| ETH | pullback | range | 64 | −0.51% | 907 | +0.15% | −0.67% |
| ETH | pullback | up | 66 | +2.55% | 1236 | +1.22% | +1.33% |
| ETH | continuation | range | 3 | +0.64% | 907 | +0.15% | +0.48% |
| ETH | continuation | up | 57 | +1.51% | 1236 | +1.22% | +0.28% |
| SOL | pullback | range | 34 | −1.44% | 545 | +0.33% | −1.78% |
| SOL | pullback | up | 35 | +1.35% | 803 | +2.59% | −1.24% |
| SOL | continuation | range | 2 | +3.71% | 545 | +0.33% | +3.38% |
| SOL | continuation | up | 33 | +8.86% | 803 | +2.59% | +6.27% |

No episode of either setup started in a "down" trend regime.

### Backtests (0.15% per side)

| Asset | Strategy | Total return | Annual volatility | Max drawdown | Trades | Time in market | Win rate | Avg trade |
|---|---|---|---|---|---|---|---|---|
| ETH | continuation 5d | +129.1% | 25.4% | −28.2% | 51 | 7.8% | 56.9% | +2.1% |
| ETH | continuation 10d | +33.2% | 34.4% | −55.0% | 39 | 12.0% | 51.3% | +2.0% |
| ETH | pullback 5d | +46.8% | 40.4% | −66.1% | 139 | 21.4% | 53.2% | +0.8% |
| ETH | pullback 10d | +573.3% | 48.9% | −67.6% | 94 | 28.9% | 48.9% | +3.5% |
| ETH | buy and hold | +413.5% | 85.3% | −94.0% | 1 | 100% | — | — |
| SOL | continuation 5d | +1,289.8% | 35.9% | −19.1% | 29 | 6.7% | 75.9% | +10.3% |
| SOL | continuation 10d | +2,061.9% | 44.2% | −29.9% | 23 | 10.6% | 69.6% | +16.6% |
| SOL | pullback 5d | −40.6% | 46.3% | −81.3% | 75 | 17.3% | 41.3% | −0.1% |
| SOL | pullback 10d | −39.5% | 53.3% | −64.8% | 50 | 23.1% | 48.0% | −0.2% |
| SOL | buy and hold | +4,860.0% | 109.5% | −96.3% | 1 | 100% | — | — |

### Random-entry baseline (continuation strategy, 2,000 draws)

| Asset | Hold | Random pool | Pool size | Trades | Actual avg trade | Random median | Share of random draws ≥ actual |
|---|---|---|---|---|---|---|---|
| ETH | 5d | any day | 3247 | 51 | +2.11% | +0.51% | 10.6% |
| ETH | 5d | up-trend day | 1236 | 51 | +2.11% | +1.41% | 30.8% |
| ETH | 10d | any day | 3242 | 39 | +2.00% | +1.24% | 38.2% |
| ETH | 10d | up-trend day | 1236 | 39 | +2.00% | +3.25% | 68.9% |
| SOL | 5d | any day | 2157 | 29 | +10.28% | +1.35% | 0.05% |
| SOL | 5d | up-trend day | 803 | 29 | +10.28% | +3.29% | 0.45% |
| SOL | 10d | any day | 2152 | 23 | +16.61% | +2.90% | 0.45% |
| SOL | 10d | up-trend day | 803 | 23 | +16.61% | +7.51% | 4.9% |

In [9]:
sol = add_regimes(add_features(load_processed("SOL/USDT", DATA_DIR)))
sol = sol.dropna(subset=["dist_sma_100", "rel_volume_20", "er_20"])
sol["fwd_ret_5"] = np.log(sol["close"].shift(-5) / sol["close"])

flags = setup_flag(sol, "continuation")
starts = flags & ~flags.shift(fill_value=False)
entries = sol[starts].dropna(subset=["fwd_ret_5"])

# Check 1: is the edge spread across years, compared with each year's own base rate?
by_year = entries.groupby(entries.index.year)["fwd_ret_5"].agg(episodes="size", mean_5d="mean")
by_year["base_5d"] = sol.groupby(sol.index.year)["fwd_ret_5"].mean()
by_year["excess_5d"] = by_year["mean_5d"] - by_year["base_5d"]
print(by_year.round(4))

# Check 2: how much of the backtest depends on its three best trades?
trades, _ = run_backtest(sol, flags, 5, 0.0015)
growth = np.log1p(trades["net_return"])
top3_share = growth.nlargest(3).sum() / growth.sum()
print(f"trades: {len(trades)} | share of total log growth from the 3 best trades: {top3_share:.0%}")
print("without the 3 best trades, avg trade:",
      round(trades["net_return"].drop(trades["net_return"].nlargest(3).index).mean(), 4))

      episodes  mean_5d  base_5d  excess_5d
date                                       
2021        11   0.1610   0.0603     0.1008
2022         1   0.0896  -0.0360     0.1256
2023        11   0.0822   0.0293     0.0529
2024         4   0.0336   0.0096     0.0240
2025         5   0.0181  -0.0063     0.0244
2026         3   0.0021  -0.0041     0.0062
trades: 29 | share of total log growth from the 3 best trades: 38%
without the 3 best trades, avg trade: 0.0681
